In [1]:
!pip install h5py

  Using cached h5py-3.11.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.5 kB)
Using cached h5py-3.11.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (5.3 MB)


In [1]:
import random
random.seed(1234)
import pandas as pd
import numpy as np
import gc
gc.collect()
from sklearn.model_selection import train_test_split
import os
import math
import torch
torch.manual_seed(123)
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer, TransformerDecoderLayer, TransformerDecoder
from joblib import Parallel, delayed
import multiprocessing
from io import open
import argparse
import time
import math
import torch.onnx
import h5py
import os 
import pickle
from multiprocessing import cpu_count, Pool
from datetime import datetime
from google.cloud import storage
import joblib
from io import BytesIO
import google.auth
from google.auth import impersonated_credentials
from datetime import datetime
import pytz

import warnings
warnings.filterwarnings("ignore")

In [2]:
input_table_name = 'anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_OOT_o3_score_ending'  # change to yours

output_feature_table_name = 'anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_OOT_'  # change to yours

number_of_parts_2_run = 18# of parts to break the input table to.  In case Vertex AI fails, you can restart from where it fails

In [3]:
from google.cloud import bigquery
client = bigquery.Client()

credentials, project= google.auth.default()
print('credentials:', credentials, ', project:', project)

api_client = bigquery.Client(credentials=credentials)

job_labels = {"costcenter": 13070, "owner": "palmere1_aetna_com"}
load_config = bigquery.LoadJobConfig(labels=job_labels)

credentials: <google.oauth2.credentials.Credentials object at 0x7fe441e072e0> , project: anbc-hcb-dev


In [4]:
class TransformerModel(nn.Module):
    def __init__(self, nhead, nhid, nlayers, dropout=0.05):
        super(TransformerModel, self).__init__()
        self.embedding_cd = nn.Embedding(cd_cnt,embedding_size)
        self.embedding_cd.weight.requires_grad = True
        self.embedding_gender_cd = nn.Embedding(4,embedding_size)
        self.embedding_gender_cd.weight.requires_grad = True
        self.embedding_age_in_months = nn.Embedding(1440,embedding_size)  
        self.embedding_age_in_months.weight.requires_grad = True
        encoder_layers_cd = TransformerEncoderLayer(embedding_size, 4, embedding_size, 0)
        self.transformer_encoder_cd = TransformerEncoder(encoder_layers_cd, 1)        
        encoder_layers_dy = TransformerEncoderLayer(embedding_size, nhead, nhid, dropout)
        self.transformer_encoder_dy = TransformerEncoder(encoder_layers_dy, nlayers)
 
        self.mm = nn.GELU()
        self.decoder_cd = nn.Linear(embedding_size, target_cd_cnt)
        self.dropout = nn.Dropout(0.1)
        self.norm = nn.LayerNorm(embedding_size)
        self.init_weights()
 
    def _generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask
 
    def init_weights(self):
        initrange = 0.1
        nn.init.zeros_(self.decoder_cd.weight)
        nn.init.uniform_(self.decoder_cd.weight, -initrange, initrange)       
    def forward(self, x):     
        gpu_batchsize = x.shape[0]
        age_in_months = x[:,:,0]
        gender_cd = x[:,:,1]
        gender_cd = self.embedding_gender_cd(gender_cd)
        age_in_months = self.embedding_age_in_months(age_in_months)
        cd = x[:,:,2:]
        cd = self.embedding_cd(cd)
        cd_res = cd.sum(-2)
        # print(cd_res.shape)
        cd = cd.reshape(gpu_batchsize*len_dy,len_cd,embedding_size)
        cd = torch.swapaxes(cd, 0, 1) 
        cd = self.transformer_encoder_cd(cd)
        cd = cd.permute(1,2,0)
        cd = nn.MaxPool1d(len_cd)(cd)
        cd = cd.reshape(gpu_batchsize,len_dy,embedding_size)
        # print(cd.shape)
        cd = cd_res+cd + gender_cd + age_in_months
        # print(cd.shape)
        cd = self.mm(cd)
        cd = self.norm(cd)
        cd = torch.swapaxes(cd, 0, 1)
 
        mth_mask = self._generate_square_subsequent_mask(len_dy).to(device)      
        cd = self.transformer_encoder_dy(cd, mth_mask)
        cd = torch.swapaxes(cd, 0, 1)
        cd = self.norm(cd)
        cd = self.dropout(cd)
 
        cd = self.decoder_cd(cd)
        cd = F.log_softmax(cd, dim=-1)
 
        return cd

In [5]:
def dataLoader(fileid):
    blob = storage.Client(credentials=google.auth.default()[0]).bucket(bucket_name).blob(fileid)
    data = BytesIO()
    blob.download_to_file(data)
    data=joblib.load(data)
    return data

In [6]:
def currentTime():
    newYorkTz = pytz.timezone("America/New_York") 
    timeInNewYork = datetime.now(newYorkTz)
    currentTimeInNewYork = timeInNewYork.strftime("%D %H:%M:%S")
    return currentTimeInNewYork
 
def conv_cd(ipt):
    ipt = ipt.split('*')
    ipt = ipt[:len_dy]
    ipt = ipt + (len_dy-len(ipt))*['']
    ipt = [dy.split(',') for dy in ipt]
    ipt = [[int(cd) if cd!='' else 0 for cd in dy] for dy in ipt]
    ipt = [dy + (len_cd-len(dy))*[0] for dy in ipt]
    return ipt
 
def conv_age_gender(ipt):
    ipt = ipt.split('*')
    ipt = ipt[:len_dy]
    ipt = [min(int(cd),1439) for cd in ipt]
    ipt = ipt + (len_dy-len(ipt))*[0]
    return ipt
 
def conv_dy(x):
    x = x.split('*')
    x = x[:len_dy]
    x = [int(cd) for cd in x]
    return x

In [7]:
def prepare_tensor(batch):
    age_in_months = [conv_age_gender(ipt) for ipt in batch['age_in_months'].tolist()]
    age_in_months = torch.tensor(age_in_months).to(device)
    age_in_months = age_in_months.reshape(batch_size,len_dy,1)
    gender_cd = [conv_age_gender(ipt) for ipt in batch['gender_cd'].tolist()]
    gender_cd = torch.tensor(gender_cd).to(device)
    gender_cd = gender_cd.reshape(batch_size,len_dy,1)    
    cd = [conv_cd(ipt) for ipt in batch['cd'].tolist()]
    cd = torch.tensor(cd).to(device)
    x = torch.cat([age_in_months,gender_cd,cd],dim=-1)
 
    dt_cnt = batch['dt_cnt'].tolist()
 
    return dt_cnt,x
def train(data):
    model.train()
    nbatch = int(data.shape[0]/batch_size)
    for i in range(nbatch):
        if i%1000 == 0:
            print('batch',i,currentTime())
        optimizer.zero_grad()
        batch = data.iloc[i*batch_size:i*batch_size+batch_size,:]
        dt_cnt,x,y = prepare_tensor(batch)
        opt = model(x)
        opt = opt.reshape(batch_size*len_dy,target_cd_cnt)
        y = [item for sublist in y for item in sublist]
        opt = torch.cat([opt[len_dy*i:len_dy*i+dt_cnt[i],:] for i in range(batch_size)],dim=0)
        y = torch.tensor(y).to(device)
        loss = criterion(opt, y)        
        loss.backward()
        optimizer.step()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
        for p in model.parameters():
            p.data.add_(p.grad, alpha=-optimizer.param_groups[0]['lr'])
 
        del batch,x,y,opt,loss
        gc.collect()
        torch.cuda.empty_cache()

In [8]:
def score(data,filename):
    model.eval()
    activation = {}
    def get_activation(name):
        def hook(model, input, output):
            activation[name] = output.detach()
        return hook
 
    model.transformer_encoder_dy.register_forward_hook(get_activation('transformer_encoder_dy'))
 
    dsize = data.shape[0]
    nbatch = int(dsize/batch_size)
    if dsize-nbatch*batch_size>0:
        k = batch_size - (dsize-nbatch*batch_size) # fill some records to build last trunk; these will be deleted in the end
        data = pd.concat([data,data.head(k)])
    data = data.reset_index(drop=True)
    nbatch = int(data.shape[0]/batch_size)
    ys = []
    for i in range(nbatch):
        batch = data.iloc[i*batch_size:i*batch_size+batch_size,:]
        dt_cnt,x = prepare_tensor(batch)
        opts = model(x)
        intermedia_output = activation['transformer_encoder_dy']       
        intermedia_output = [intermedia_output[dt_cnt[i],i,:].reshape(1,-1) for i in range(batch_size)]
        intermedia_output = torch.cat(intermedia_output)           
        ys.append(intermedia_output)
    ys = torch.cat(ys).cpu().numpy()
    ys = pd.DataFrame(ys,columns = ['emb'+str(i) for i in range(embedding_size)])
    ys[entity_id] = data[entity_id]
    ys = ys.head(dsize)
    blob = storage.Client(credentials=google.auth.default()[0]).bucket(bucket_name).blob(filename)
    with blob.open("wb", ignore_flush=True) as f:
        joblib.dump(ys, f)

In [9]:
def score_file(input_tbl, output_tbl, number_of_parts):
    #filename = os.path.join(data_path,ipt_name)
 
    #data = dataLoader(filename)
    
    sql = """
        select *
        from """ + input_tbl + """
        WHERE 1=1
        order by individual_id
    """
    data_all = client.query(sql).to_dataframe()  
    
    part_size = int(np.ceil(data_all.shape[0]/number_of_parts))
    
    print('all:', data_all.shape, ', part_size: ', part_size)
    
    for ith in range(number_of_parts):
        
        from_pos = ith * part_size 
        end_pos = (ith + 1) * part_size
        
        if ith == (number_of_parts - 1): end_pos = data_all.shape[0] +1
        
        data = data_all.iloc[from_pos : end_pos]

        print('after reading:', currentTime())

        print('input from:', from_pos, ' to:', end_pos)

        data['dt_cnt'] = data['dt_cnt'] - 1

        trunksize = batch_size*10

        ntrunks = int(data.shape[0]/trunksize) + 1

        for trunk_id in range(ntrunks):
            trunkdata = data.iloc[trunk_id*trunksize:(trunk_id+1)*trunksize,]
            trunkid = os.path.join(data_path,'trunk'+str(trunk_id)+'.p')
            score(trunkdata,trunkid)
            print(ith, ', scoring trunk_id:', trunk_id, ' of ', ntrunks)

        data2 = []
        for trunk_id in range(ntrunks):  
            trunkid = os.path.join(data_path,'trunk'+str(trunk_id)+'.p')
            data2.append(dataLoader(trunkid))
            print(ith, ', saving trunk_id:', trunk_id, ' of ', ntrunks)

        data2 = pd.concat(data2)
        
        print('after embedding:', currentTime())
        print(data2.shape)

        output_part = output_tbl + '_part_' + str(ith)
        
        api_client.delete_table(output_part + '_tmp', not_found_ok=True)

        load_job = api_client.load_table_from_dataframe(
            dataframe = data2,
            destination = output_part + '_tmp',
            job_config = load_config,
        )

        print(load_job.result())

        sql = """
            select count(1) as cnt
            from """ + output_part + '_tmp'

        cnt = client.query(sql).to_dataframe()  

        print('after:', cnt, 'output table name:', output_part + '_tmp')
        #for trunk_id in range(ntrunks):  
        #    trunkid = os.path.join(data_path,'trunk'+str(trunk_id)+'.p')
        #    blob = storage.Client(credentials=google.auth.default()[0]).bucket(bucket_name).blob(trunkid)
        #    blob.delete()

In [ ]:
%%time
print('program start:', currentTime())

bucket_name = "clin-analytics-data-hcb-dev"
# data_source = 'a321276/TransformerV9/Data/a321276_o2_'
model_path = 'a534354/TransformerV10/Model'
entity_id = 'individual_id'

# maybe further tuning the machine!!!!, 

batch_size = 16  #512, Jane changed to run on t4!!!
embedding_size = 256
minimum_mth_training = 6   #filter out data points which short month length
len_dy = 200 # how many days in th seq
len_cd = 80 # within a day how many cds. 
nhead = 16 # heads of transformer - double transformer share same feature...
nhid = 512 # number of hidden of transformer - double transformer share same feature...
nlayers = 6 # number of layers of transformer - double transformer share same feature...
ndropout = 0.1 # dropout rate of transformer - double transformer share same feature...
cd_cnt = 98041 # numbr of codes used in embedding matrix
target_cd_cnt = 2 # numbr of target codes 
criterion = nn.NLLLoss()
parallel = True
device = torch.device("cuda:0") 
#device = torch.device("cuda") #Jane changed to run on t4!!!
 
 
blob = storage.Client(credentials=google.auth.default()[0]).bucket(bucket_name).blob(os.path.join(model_path,'bestModel_singlegpu'))
bestModel = BytesIO()
blob.download_to_file(bestModel)
bestModel=joblib.load(bestModel)  
model = TransformerModel(nhead, nhid, nlayers, 0)
model.load_state_dict(bestModel['model'])
model = model.to(device)
 
data_path = 'a321276/TransformerV10/Data'
#ipt_name = 'a321276_o3_score_ending0.p'
 
#opt_name = 'tmp.p'
#score_file(data_path,ipt_name,opt_name)

score_file(input_table_name, output_feature_table_name, number_of_parts_2_run)

print('end:', currentTime())

program start: 08/19/24 10:51:59
all: (1875523, 5) , part_size:  104196
after reading: 08/19/24 10:53:11
input from: 0  to: 104196
0 , scoring trunk_id: 0  of  652
0 , scoring trunk_id: 1  of  652
0 , scoring trunk_id: 2  of  652
0 , scoring trunk_id: 3  of  652
0 , scoring trunk_id: 4  of  652
0 , scoring trunk_id: 5  of  652
0 , scoring trunk_id: 6  of  652
0 , scoring trunk_id: 7  of  652
0 , scoring trunk_id: 8  of  652
0 , scoring trunk_id: 9  of  652
0 , scoring trunk_id: 10  of  652
0 , scoring trunk_id: 11  of  652
0 , scoring trunk_id: 12  of  652
0 , scoring trunk_id: 13  of  652
0 , scoring trunk_id: 14  of  652
0 , scoring trunk_id: 15  of  652
0 , scoring trunk_id: 16  of  652
0 , scoring trunk_id: 17  of  652
0 , scoring trunk_id: 18  of  652
0 , scoring trunk_id: 19  of  652
0 , scoring trunk_id: 20  of  652
0 , scoring trunk_id: 21  of  652
0 , scoring trunk_id: 22  of  652
0 , scoring trunk_id: 23  of  652
0 , scoring trunk_id: 24  of  652
0 , scoring trunk_id: 25  of 